# Validation against Hull + live SVI pipeline demo

This notebook does two things:

1. **Validates the engine** against reference values from Hull, *Options, Futures, and Other Derivatives*: European option prices and Greeks, plus internal consistency checks (put-call parity, finite-difference cross-checks), the Crank-Nicolson solver, and implied-volatility round trips.
2. **Runs the full market-data pipeline live**: fetch a real option chain (AAPL), screen it for arbitrage, compute implied volatilities, fit an SVI smile, and plot the result.

If the machine has no internet access the live section automatically falls back to a synthetic chain so the whole notebook still runs.

Run the cells in order (Kernel -> Restart & Run All works too).

In [ ]:
import sys
from pathlib import Path

# Locate the project root and add src/ to the path.
# This works whether the notebook kernel is rooted at the project root
# (common when launching Jupyter from the terminal) or at notebooks/
# (common when VS Code sets the kernel cwd to the file's directory).
_project_root = next(
    p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if (p / 'src').is_dir()
)
sys.path.insert(0, str(_project_root / 'src'))

import numpy as np
import pandas as pd

from engine.core import all_greeks, black_scholes_price, vega
from engine.market import (
    OptionChain,
    OptionQuote,
    check_all,
    fetch_expiries,
    fetch_option_chain,
    remove_arbitrage_violations,
)
from engine.risk import crank_nicolson_price
from engine.vol import (
    fit_svi_from_chain,
    implied_volatility,
    svi_total_variance,
)

pd.set_option('display.precision', 6)
print('engine imports OK')
print('project root:', _project_root)

## 1. Black-Scholes prices vs Hull

Two worked examples from Hull. The classic one (S=42, K=40, T=0.5, r=10%, sigma=20%) prices a call at **4.7594** and a put at **0.8086**; the second (S=100, K=100, T=1, r=5%, sigma=20%) gives 10.4506 / 5.5735.

In [ ]:
book_cases = {
    (42.0, 40.0, 0.5, 0.10, 0.20): 'Hull: S=42, K=40, T=0.5, r=10%, sig=20%',
    (100.0, 100.0, 1.0, 0.05, 0.20): 'Hull: S=100, K=100, T=1, r=5%, sig=20%',
}
book_prices = {
    ('Hull: S=42, K=40, T=0.5, r=10%, sig=20%', 'call'): 4.7594,
    ('Hull: S=42, K=40, T=0.5, r=10%, sig=20%', 'put'): 0.8086,
    ('Hull: S=100, K=100, T=1, r=5%, sig=20%', 'call'): 10.4506,
    ('Hull: S=100, K=100, T=1, r=5%, sig=20%', 'put'): 5.5735,
}

rows = []
for (S, K, T, r, sig), case in book_cases.items():
    for option_type in ('call', 'put'):
        engine = float(black_scholes_price(
            S=S, K=K, T=T, r=r, sigma=sig, option_type=option_type
        ))
        hull = book_prices[(case, option_type)]
        rows.append({
            'case': case,
            'type': option_type,
            'engine': engine,
            'hull': hull,
            'abs diff': abs(engine - hull),
        })

pd.DataFrame(rows)

## 2. Greeks vs Hull

Hull's published Greek values for the S=100, K=100, T=1, r=5%, sigma=20% case (delta, gamma, vega, theta, rho for calls and puts).

In [ ]:
book_greeks = {
    'call': {'delta': 0.6368, 'gamma': 0.0188, 'vega': 37.5240, 'theta': -6.4140, 'rho': 53.2325},
    'put': {'delta': -0.3632, 'gamma': 0.0188, 'vega': 37.5240, 'theta': -1.6579, 'rho': -41.8905},
}

rows = []
for option_type, book in book_greeks.items():
    engine = all_greeks(S=100.0, K=100.0, T=1.0, r=0.05, sigma=0.20, option_type=option_type)
    for greek, expected in book.items():
        value = float(engine[greek])
        rows.append({
            'type': option_type,
            'greek': greek,
            'engine': round(value, 4),
            'hull': expected,
            'abs diff': abs(value - expected),
        })

pd.DataFrame(rows)

## 3. Internal consistency (self-checks)

Put-call parity must hold to machine precision, and the analytic Greeks must match finite-difference approximations of the price.

In [ ]:
S, K, T, r, sig = 100.0, 100.0, 1.0, 0.05, 0.20
call = black_scholes_price(S=S, K=K, T=T, r=r, sigma=sig, option_type='call')
put = black_scholes_price(S=S, K=K, T=T, r=r, sigma=sig, option_type='put')

parity_rhs = S - K * np.exp(-r * T)
print('put-call parity diff   :', float(call - put - parity_rhs))

h = 1e-5
fd_delta = (
    black_scholes_price(S=S + h, K=K, T=T, r=r, sigma=sig, option_type='call')
    - black_scholes_price(S=S - h, K=K, T=T, r=r, sigma=sig, option_type='call')
) / (2 * h)
fd_vega = (
    black_scholes_price(S=S, K=K, T=T, r=r, sigma=sig + h, option_type='call')
    - black_scholes_price(S=S, K=K, T=T, r=r, sigma=sig - h, option_type='call')
) / (2 * h)

analytic = all_greeks(S=S, K=K, T=T, r=r, sigma=sig, option_type='call')
print('FD delta vs analytic   :', round(float(fd_delta), 6), 'vs', round(float(analytic['delta']), 6))
print('FD vega  vs analytic   :', round(float(fd_vega), 6), 'vs', round(float(vega(S=S, K=K, T=T, r=r, sigma=sig, option_type='call')), 6))

## 4. Crank-Nicolson vs the closed form

The finite-difference solver must reproduce the Black-Scholes formula for European options (it is also capable of American exercise, which the formula cannot price).

In [ ]:
rows = []
for S, K, T, sig in [(100.0, 100.0, 1.0, 0.20), (90.0, 100.0, 0.5, 0.30), (110.0, 100.0, 2.0, 0.15)]:
    for option_type in ('call', 'put'):
        exact = float(black_scholes_price(S=S, K=K, T=T, r=0.05, sigma=sig, option_type=option_type))
        cn = crank_nicolson_price(S=S, K=K, T=T, r=0.05, sigma=sig, option_type=option_type)
        rows.append({
            'S': S, 'K': K, 'T': T, 'sig': sig, 'type': option_type,
            'black-scholes': exact,
            'crank-nicolson': round(cn, 6),
            'rel err': abs(cn - exact) / exact,
        })

pd.DataFrame(rows)

## 5. Implied volatility round trip

Price an option with a known volatility, then invert the price back to volatility: the two must agree.

In [ ]:
rows = []
for K in (90.0, 100.0, 110.0):
    for sig in (0.10, 0.25):
        price = black_scholes_price(S=100.0, K=K, T=1.0, r=0.05, sigma=sig, option_type='call')
        iv = implied_volatility(
            S=100.0, K=K, T=1.0, r=0.05, market_price=price, option_type='call'
        )
        rows.append({'K': K, 'input sigma': sig, 'implied vol': round(iv, 8), 'abs diff': abs(iv - sig)})

pd.DataFrame(rows)

## 6. Live SVI pipeline demo

Fetch a real AAPL chain, screen it for arbitrage, compute implied volatilities, and fit an SVI smile. Falls back to a synthetic chain (generated from a known SVI curve) when the network is unavailable, so the demo always runs.

In [ ]:
Q = 0.005  # approximate continuous dividend yield for AAPL

try:
    expiries = fetch_expiries('AAPL')
    chain = fetch_option_chain('AAPL', str(expiries[0]))
    source = 'live'
    print('live chain:', chain.symbol, chain.expiry, 'spot', round(chain.spot, 2), 'quotes', len(chain.quotes))
except Exception as exc:
    print('live fetch failed (' + type(exc).__name__ + '); falling back to a synthetic chain')
    source = 'synthetic'
    from datetime import date

    spot = 100.0
    strikes = np.arange(80.0, 121.0, 5.0)
    true_params = dict(a=0.04, b=0.4, rho=-0.6, m=0.1, sigma=0.15)
    as_of, expiry = date(2025, 1, 16), date(2026, 1, 16)
    quotes = []
    for strike in strikes:
        k = np.log(strike / spot)
        vol = float(np.sqrt(svi_total_variance(k, **true_params) / 1.0))
        for option_type in ('call', 'put'):
            mid = black_scholes_price(S=spot, K=float(strike), T=1.0, r=0.04, q=Q, sigma=vol, option_type=option_type)
            quotes.append(OptionQuote(
                symbol='SYNTH', expiry=expiry, strike=float(strike),
                option_type=option_type, bid=mid, ask=mid,
            ))
    chain = OptionChain(symbol='SYNTH', as_of=as_of, spot=spot, expiry=expiry, quotes=quotes)
    print('synthetic chain ready')

In [ ]:
R = 0.04

violations = check_all(chain, r=R, q=Q, tolerance=0.05)
print('arbitrage violations (tolerance 0.05):', len(violations))
for violation in violations[:5]:
    print('  ', violation)

clean = remove_arbitrage_violations(chain, r=R, q=Q, tolerance=0.05)
print('quotes kept:', len(clean.quotes))

fit = fit_svi_from_chain(chain=clean, r=R, q=Q)
print('SVI parameters:', dict(zip(('a', 'b', 'rho', 'm', 'sigma'), [round(float(x), 5) for x in fit.params])))
print('sum of squared residuals:', round(fit.cost, 6), '| fitted points:', fit.n_observations)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

T = clean.time_to_expiry()
ks = np.linspace(
    min(np.log(q.strike / clean.spot) for q in clean.quotes),
    max(np.log(q.strike / clean.spot) for q in clean.quotes),
    200,
)
smile = fit.implied_volatility(ks, T=T)

plt.figure(figsize=(8, 5))
for option_type in ('call', 'put'):
    quotes = [q for q in clean.quotes if q.option_type == option_type]
    ks_data = np.array([np.log(q.strike / clean.spot) for q in quotes])
    vols_data = np.array([
        implied_volatility(
            S=clean.spot, K=q.strike, T=T, r=R, q=Q,
            market_price=q.mid, option_type=q.option_type,
        )
        for q in quotes
    ])
    plt.scatter(ks_data, vols_data, s=25, label=option_type + ' implied vols')
plt.plot(ks, smile, 'k-', linewidth=2, label='SVI fit')
plt.xlabel('log-moneyness ln(K/S)')
plt.ylabel('implied volatility')
plt.title('Implied volatility smile (' + source + ' chain)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print('fitted ATM volatility (k=0):', round(float(fit.implied_volatility(0.0, T=T)), 4))

## Summary

If the tables above show tiny or zero differences, the engine reproduces Hull's reference values and is internally consistent: Black-Scholes prices and Greeks match the book, the Crank-Nicolson solver agrees with the closed form, implied-volatility inversion round-trips, and the full pipeline (fetch, filter, implied vol, SVI fit) runs end to end.

Note on live data: real quotes are American options quoted with bid/ask spreads, so put-call parity and the other checks use a tolerance (0.05 here) and the fitted SVI curve is an approximation of the market smile, not an exact reproduction.